In [ ]:
from torch import float32, no_grad, tensor, randn, zeros
from torch.autograd import Function
from torch.nn import Linear, Module, Parameter

import import_ipynb
from common import assert_eq, T # type: ignore

$$ f=x+1 $$
$$ \frac{\partial f}{\partial x} = 1 $$
$$ \diamond \diamond \diamond $$
$$ \text{Let } F=3+f $$
$$ \frac{\partial F}{\partial f} = 1 $$
$$ \frac{\partial F}{\partial x} = \frac{\partial F}{\partial f} \cdot \frac{\partial f}{\partial x} = 1$$

In [ ]:
class Add1Function(Function):
    @staticmethod
    def forward(ctx, x):
        return x + 1

    # Demonstration version #1
    @staticmethod
    def backward_1(ctx, dF_df):
        # For this example only
        # assert_eq(dF_df.shape, (1,))
        # assert_eq(dF_df.item(), 1.0)

        df_dx = 1
        dF_dx = dF_df * df_dx

        return (dF_dx, )
   
    @staticmethod
    def backward(ctx, _):
        return (T(1), )


class Add1(Module):
    def forward(self, x):
        return Add1Function.apply(x)
   

def _test_Add1():
    x = tensor(4, dtype=float32, requires_grad=True)
    F = 3 + Add1()(x)     
    F.backward()     

    assert_eq(F.item(), 3.0 + (4.0 + 1))
    assert_eq(x.grad.item(), 1.0)


if __name__ == "__main__":
    _test_Add1()

$$ f=2x $$
$$ \frac{\partial f}{\partial x} = 2 $$
$$ \diamond \diamond \diamond $$
$$ \text{Let } F=1+3f $$
$$ \frac{\partial F}{\partial f} = 3 $$
$$ \frac{\partial F}{\partial x} = \frac{\partial F}{\partial f} \cdot \frac{\partial f}{\partial x} $$

In [ ]:
class Mul2Function(Function):
    @staticmethod
    def forward(ctx, x):
        return 2 * x

    # Demonstration version #1
    @staticmethod
    def backward_1(ctx, dF_df):
        # For this example only
        # assert_eq(dF_df.shape, (1,))
        # assert_eq(dF_df.item(), 3.0)
        
        df_dx = 2
        dF_dx = dF_df * df_dx

        return (dF_dx, )
   
    @staticmethod
    def backward(ctx, _) :
        return (T([3 * 2]), )


class Mul2(Module):
    def forward(self, x):
        return Mul2Function.apply(x)
   

def _test_Mul2():
    x = tensor([4.0], requires_grad=True)
    F = 1 + 3 * Mul2()(x)     
    F.backward()     

    assert_eq(F.item(), 1.0 + 3.0 * (2.0 * 4.0))
    assert_eq(x.grad.item(), 3.0 * 2.0)


if __name__ == "__main__":
    _test_Mul2()

$$ f=xy $$
$$ \frac{\partial f}{\partial x} = y $$
$$ \frac{\partial f}{\partial y} = x $$
$$ \diamond \diamond \diamond $$
$$ \text{Let } F=2+5f $$
$$ \frac{\partial F}{\partial f} = 5 $$
$$ \frac{\partial F}{\partial x} = \frac{\partial F}{\partial f} \cdot \frac{\partial f}{\partial x} $$
$$ \frac{\partial F}{\partial y} = \frac{\partial F}{\partial f} \cdot \frac{\partial f}{\partial y} $$

In [ ]:
class MultiplyFunction(Function):
    @staticmethod
    def forward(ctx, x, y):
        ctx.save_for_backward(x, y)
        return x * y

    @staticmethod
    def backward(ctx, dF_df):
        # For this example only
        # assert_eq(dF_df.shape, (1,))
        # assert_eq(dF_df.item(), 5.0)

        (x, y) = ctx.saved_tensors

        df_dx = y
        dF_dx =  dF_df * df_dx

        df_dy = x
        dF_dy =  dF_df * df_dy

        return (dF_dx, dF_dy)
   

class Multiply(Module):
    def forward(self, x, y):
        return MultiplyFunction.apply(x, y)
   

def _test_MultiplyFunction():
    x = tensor([3.0], requires_grad=True)
    y = tensor([4.0], requires_grad=True)

    F = 2.0 + 5.0 * Multiply()(x, y)     
    F.backward()        

    assert_eq(F.item(), 2.0 + 5.0 * (3.0 * 4.0))
    assert_eq(x.grad.item(), 4.0 * 5.0)
    assert_eq(y.grad.item(), 3.0 * 5.0)


if __name__ == "__main__":
    _test_MultiplyFunction()

$$
sigmoid(x) = f =
\frac{e^x}{e^x+1} =  
\frac{e^x}{e^x} \cdot \frac{1}{1+e^{-x}}
$$

$$ \frac{\partial f}{\partial x} = \frac{e^x}{(e^x+1)^2} = f(1-f) $$

$$ \diamond \diamond \diamond $$

$$ \text{Let } F=2f+3 $$
$$ \frac{\partial F}{\partial f} = 2 $$
$$ \frac{\partial F}{\partial x} = \frac{\partial F}{\partial f} \cdot \frac{\partial f}{\partial x} $$

In [ ]:
class SigmoidFunction(Function):
    @staticmethod
    def forward(ctx, x):
        sigmoid_x = 1 / (1 + (-x).exp())
        ctx.save_for_backward(sigmoid_x)
        return sigmoid_x

    @staticmethod
    def backward(ctx, dF_df):
        # For this example only:
        # assert_eq(dF_df.shape, (1,))
        # assert_eq(dF_df.item(), 2.0)
        
        (sigmoid_x, ) = ctx.saved_tensors

        df_dx = sigmoid_x * (1 - sigmoid_x)
        dF_dx = dF_df * df_dx

        return (dF_dx, )
    

class Sigmoid(Module):
    def forward(self, x):
        return SigmoidFunction.apply(x)
    

def _test_SigmoidFunction():
    x = tensor([0.0], requires_grad=True)

    F = 2 * Sigmoid()(x) + 1
    F.backward()     

    assert_eq(F.item(), 2.0, atol=0.001)
    assert_eq(x.grad.item(), 0.5, atol=0.001)


if __name__ == "__main__":
    _test_SigmoidFunction()


$$ f = xW + b $$
$$ \frac{\partial f}{\partial x} = W $$
$$ \frac{\partial f}{\partial W} = x $$
$$ \frac{\partial f}{\partial b} = 1 $$
$$ \diamond \diamond \diamond $$
$$ \text{Let } F=f $$
$$ \frac{\partial F}{\partial f} = 1 $$
$$ \frac{\partial F}{\partial x} = \frac{\partial F}{\partial f} \cdot \frac{\partial f}{\partial x} $$
$$ \frac{\partial F}{\partial W} = \frac{\partial F}{\partial f} \cdot \frac{\partial f}{\partial W} $$
$$ \frac{\partial F}{\partial b} = \frac{\partial F}{\partial f} \cdot \frac{\partial f}{\partial b} $$

In [ ]:
class LinearFunction(Function):
    @staticmethod
    def forward(ctx, x, W, b):
        ctx.save_for_backward(x, W, b)
        return x @ W + b

    @staticmethod
    def backward(ctx, dF_df):
        x, W, b = ctx.saved_tensors
        (samples, features, output) = (x.shape[0], x.shape[1], W.shape[1])
        assert_eq(x.shape, (samples, features))
        assert_eq(W.shape, (features, output))
        assert_eq(b.shape, (output,))

        f = x @ W + b
        assert_eq(f.shape, (samples, output))

        F = f
        assert_eq(F.shape, (samples, output))

        dF_df_shape = dF_df.shape
        assert_eq(dF_df_shape, (samples, output))

        f_shape = (x @ W + b).shape
        assert_eq(f_shape, (samples, output))

        F_shape = F.shape
        assert_eq(F_shape, (samples, output))

        df_dx = W
        assert_eq(df_dx.shape, (features, output))
        
        # (samples, output) @ (output, features) -> (samples, features)
        dF_dx = dF_df @ df_dx.T
        assert_eq(dF_dx.shape, x.shape)
        assert_eq(dF_dx.shape, (samples, features))
        
        df_dW = x
        assert_eq(df_dW.shape, (samples, features))

        # (features, samples) @ (samples, output) -> (features, output)
        dF_dW = df_dW.T @ dF_df          
        assert_eq(dF_dW.shape, W.shape)
        assert_eq(dF_dW.shape, (features, output))
        
        # (output,) * (output,) -> (output, )
        df_db = 1
        dF_db = df_db * dF_df.sum(dim=0)     
        assert_eq(dF_db.shape, b.shape)
        assert_eq(dF_db.shape, (output,))
        
        return (dF_dx, dF_dW, dF_db)
    

class Linear(Module):
    def __init__(self, in_features, out_features):
        super().__init__()

        self.W = Parameter(randn(in_features, out_features) * 0.01)
        self.b = Parameter(zeros(out_features))

    def forward(self, x):
        return LinearFunction.apply(x, self.W, self.b)
    

def _test_LinearFunction():
    x = tensor([[1, 2, 3],
                [6, 5, 4],
                [7, 9, 8]], dtype=float32, requires_grad=True)
    
    layer = Linear(3, 2)

    F = layer(x).sum()
    F.backward()


if __name__ == "__main__":
    _test_LinearFunction()


$$ BCE = f(y) = -(y\ln(p)+(1-y)\ln(1-p)) $$
$$ \frac{\partial f}{\partial p} = \frac{p-y}{p(1-p)} $$
$$ \frac{\partial f}{\partial y} = N1 $$
$$ \text{Let } F=f $$
$$ \frac{\partial F}{\partial f} = 1 $$
$$ \frac{\partial F}{\partial p} = \frac{\partial F}{\partial f} \cdot \frac{\partial f}{\partial p} $$

In [ ]:
class BinaryCrossEntropyFunction(Function):
    @staticmethod
    def forward(ctx, p, y):
        p = p.clamp(0.0001, 0.9999)

        ctx.save_for_backward(p, y)
        bce = - (y * p.log() + (1 - y) * (1 - p).log())

        return bce.mean()

    @staticmethod
    def backward(ctx, dF_df):
        (p, y) = ctx.saved_tensors

        df_dp = (p - y) / (p * (1 - p))
        dF_dp = dF_df * df_dp / p.size(0)
        dF_dy = N1

        return (dF_dp, dF_dy)
   

class BinaryCrossEntropy(Module):
    def __init__(self):
        super().__init__()

    def forward(self, p, y):
        return BinaryCrossEntropyFunction.apply(p, y)
   

def _test_bce():
    p = tensor([[1.0],[0.0],[0.0]], requires_grad=True)
    y = tensor([[1.0],[0.0],[0.0]])

    F = BinaryCrossEntropy()(p, y)
    F.backward()

    assert_eq(F.item(), 0.0001, atol=0.001)
    assert_eq(p.grad[0].item(), -0.3333, atol=0.001)
    assert_eq(p.grad[1].item(), +0.3333, atol=0.001)
    assert_eq(p.grad[2].item(), +0.3333, atol=0.001)


if __name__ == "__main__":
    _test_bce()